# Integration Test — Phase 1 & Phase 2
## Run this notebook completely before the faculty meeting
## Every cell must print PASS or FAIL

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python /content/drive/MyDrive/Project/disaSense_project/src/models/baseline_a_cnn.py


Using device: cuda
GPU: Tesla T4
[TRAIN] Built index: 38727 buildings across 2239 image pairs.
[VAL] Built index: 5246 buildings across 279 image pairs.
Train split summary: {'split': 'train', 'n_image_pairs': 2239, 'n_buildings': 38727, 'class_distribution': {'no-damage': 28251, 'major-damage': 5027, 'minor-damage': 3468, 'destroyed': 1981}, 'disaster_type_distribution': {'socal-fire': 3986, 'guatemala-volcano': 37, 'hurricane-michael': 5955, 'palu-tsunami': 2418, 'midwest-flooding': 1504, 'santa-rosa-wildfire': 4573, 'hurricane-matthew': 1224, 'hurricane-florence': 1814, 'hurricane-harvey': 9354, 'mexico-earthquake': 7862}, 'skipped_patches': 0}
Val split summary: {'split': 'val', 'n_image_pairs': 279, 'n_buildings': 5246, 'class_distribution': {'major-damage': 606, 'no-damage': 3808, 'minor-damage': 452, 'destroyed': 380}, 'disaster_type_distribution': {'hurricane-michael': 756, 'hurricane-florence': 134, 'santa-rosa-wildfire': 695, 'hurricane-harvey': 1289, 'palu-tsunami': 345, 'me

In [ ]:
import os
import sys
import yaml
import torch
import random
import numpy as np
import p andas as pd
from IPython.display import display, Image

# Set seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Mount drive (Colab specific)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/Project/disaSense_project/notebooks')
except ImportError:
    print("Not running in Colab, skipping drive mount.")

# Add project root to path for imports
sys.path.append(os.path.abspath('..'))

try:
    with open('../configs/config.yaml') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    print("Warning: config.yaml not found at ../configs/config.yaml")
    config = {"paths": {}}

TEST_RESULTS = []
def record_result(test_num, desc, passed, msg=""):
    res = "PASS" if passed else "FAIL"
    print(f"TEST {test_num} {res}: {msg}")
    TEST_RESULTS.append({"Test": test_num, "Description": desc, "Result": res, "Message": msg})


In [ ]:
def test_1():
    desc = "Project structure check"
    try:
        # Running from notebooks/ folder, project root is '..'
        root = '..'
        expected_files = [
            "configs/config.yaml",
            "configs/lora_config.json",
            "src/preprocessing/__init__.py",
            "src/preprocessing/patch_cropper.py",
            "src/preprocessing/filter_disasters.py",
            "src/preprocessing/event_splitter.py",
            "src/models/__init__.py",
            "src/models/baseline_a_cnn.py",
            "src/training/__init__.py",
            "src/inference/__init__.py",
            "src/evaluation/__init__.py",
            "src/evaluation/visualise.py",
            "requirements.txt",
            "README.md"
        ]
        missing = []
        for f in expected_files:
            if not os.path.exists(os.path.join(root, f)):
                missing.append(f)

        if not missing:
            record_result(1, desc, True, "All project files exist")
        else:
            record_result(1, desc, False, f"Missing files: {missing}")
    except Exception as e:
        record_result(1, desc, False, str(e))

test_1()


In [ ]:
def test_2():
    desc = "xBD dataset path check"
    try:
        xbd_root = config.get("paths", {}).get("xbd_root", "")
        if not os.path.exists(xbd_root):
            record_result(2, desc, False, f"XBD_ROOT does not exist at {xbd_root}")
            return

        images_dir = os.path.join(xbd_root, "images")
        labels_dir = os.path.join(xbd_root, "labels")
        targets_dir = os.path.join(xbd_root, "targets")

        missing_dirs = []
        for d in [images_dir, labels_dir]:
            if not os.path.exists(d):
                missing_dirs.append(d)

        if missing_dirs:
            record_result(2, desc, False, f"Missing directories: {missing_dirs}")
            return

        num_png = len([f for f in os.listdir(images_dir) if f.endswith('.png')])
        num_json = len([f for f in os.listdir(labels_dir) if f.endswith('.json')])

        print(f"Found {num_png} PNG files and {num_json} JSON files.")
        if num_png > 0 and num_png >= num_json:
            record_result(2, desc, True, "xBD dataset paths and file counts look reasonable")
        else:
            record_result(2, desc, False, "File counts suspect (0 files or less images than labels)")

    except Exception as e:
        record_result(2, desc, False, str(e))

test_2()


In [ ]:
def test_3():
    desc = "Patch cropper module"
    try:
        from src.preprocessing.patch_cropper import crop_patch, parse_xbd_label, XBDPatchDataset

        xbd_root = config.get("paths", {}).get("xbd_root", "")
        dataset = XBDPatchDataset(xbd_root=xbd_root, split="train", seed=SEED)

        if len(dataset) == 0:
            record_result(3, desc, False, "Dataset is empty. Cannot test crop_patch fully.")
            return

        # Test get_class_weights
        weights = dataset.get_class_weights()
        if len(weights) != 4:
            record_result(3, desc, False, "get_class_weights() did not return length 4")
            return

        # Test get_split_summary
        summary = dataset.get_split_summary()
        required_keys = ["split", "n_image_pairs", "n_buildings", "class_distribution", "disaster_type_distribution", "skipped_patches"]
        for k in required_keys:
            if k not in summary:
                record_result(3, desc, False, f"Summary missing key: {k}")
                return

        # Test one sample
        pre, post, label, meta = dataset[0]
        if list(pre.shape) != [3, 224, 224] or list(post.shape) != [3, 224, 224]:
            record_result(3, desc, False, f"Patch shapes incorrect: pre {pre.shape}, post {post.shape}")
            return

        if not (0 <= label <= 3):
            record_result(3, desc, False, f"Label out of range: {label}")
            return

        for k in ["damage_class", "disaster_type", "bbox"]:
            if k not in meta:
                record_result(3, desc, False, f"Meta missing key: {k}")
                return

        record_result(3, desc, True, "crop_patch() and XBDPatchDataset work correctly")

    except Exception as e:
        record_result(3, desc, False, str(e))

test_3()


In [ ]:
def test_4():
    desc = "ResNet50 architecture"
    try:
        from src.models.baseline_a_cnn import ResNet50DamageClassifier
        model = ResNet50DamageClassifier(num_classes=4)

        # Test forward pass
        dummy_input = torch.randn(4, 3, 224, 224)
        out = model(dummy_input)
        if list(out.shape) != [4, 4]:
            record_result(4, desc, False, f"Forward pass shape incorrect: {out.shape}")
            return

        # Test unfreezing
        params_initial = model.get_num_trainable_params()
        print(f"Initial trainable params: {params_initial}")
        if params_initial <= 0:
            record_result(4, desc, False, "Initial trainable params should be positive")
            return

        model.unfreeze_backbone("none")
        params_none = model.get_num_trainable_params()
        if params_none != params_initial:
            record_result(4, desc, False, "Unfreeze 'none' changed param count")
            return

        model.unfreeze_backbone("last2")
        params_last2 = model.get_num_trainable_params()
        print(f"Trainable params after 'last2': {params_last2}")

        if params_last2 <= params_initial:
            record_result(4, desc, False, "Unfreeze 'last2' did not increase param count")
            return

        record_result(4, desc, True, "Model architecture correct")
    except Exception as e:
        record_result(4, desc, False, str(e))

test_4()


In [ ]:
def test_5():
    desc = "Training smoke test"
    try:
        from src.models.baseline_a_cnn import ResNet50DamageClassifier, save_checkpoint
        from src.preprocessing.patch_cropper import XBDPatchDataset
        import torch.nn as nn

        xbd_root = config.get("paths", {}).get("xbd_root", "")
        dataset = XBDPatchDataset(xbd_root=xbd_root, split="train", seed=SEED)

        if len(dataset) < 4:
            record_result(5, desc, False, "Not enough data for training smoke test.")
            return

        loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ResNet50DamageClassifier().to(device)
        model.train()

        optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss().to(device)

        losses = []
        for i, batch in enumerate(loader):
            if i >= 5:
                break
            pre, post, labels, meta = batch
            post, labels = post.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(post)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        if len(losses) == 0:
            record_result(5, desc, False, "No steps executed.")
            return

        print(f"Initial loss: {losses[0]:.4f}")
        print(f"Loss after {len(losses)} steps: {losses[-1]:.4f}")

        if np.isnan(losses[-1]) or np.isinf(losses[-1]):
            record_result(5, desc, False, "Loss is NaN or Inf")
            return

        chkpt_dir = config.get("paths", {}).get("checkpoint_dir", "../checkpoints")
        os.makedirs(chkpt_dir, exist_ok=True)
        chkpt_path = os.path.join(chkpt_dir, "smoke_test_chkpt.pt")
        save_checkpoint(model, optimizer, 1, {"loss": losses[-1]}, chkpt_path)

        if not os.path.exists(chkpt_path):
            record_result(5, desc, False, "Checkpoint file was not created.")
            return

        record_result(5, desc, True, "Training smoke test passed")
    except Exception as e:
        record_result(5, desc, False, str(e))

test_5()


In [ ]:
def test_6():
    desc = "Evaluation function"
    try:
        from src.models.baseline_a_cnn import ResNet50DamageClassifier, load_checkpoint, evaluate_baseline_a
        from src.preprocessing.patch_cropper import XBDPatchDataset
        import torch

        chkpt_dir = config.get("paths", {}).get("checkpoint_dir", "../checkpoints")
        chkpt_path = os.path.join(chkpt_dir, "smoke_test_chkpt.pt")

        if not os.path.exists(chkpt_path):
            record_result(6, desc, False, "Checkpoint from test 5 missing.")
            return

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ResNet50DamageClassifier().to(device)
        model, _, _, _ = load_checkpoint(chkpt_path, model)

        xbd_root = config.get("paths", {}).get("xbd_root", "")
        val_dataset = XBDPatchDataset(xbd_root=xbd_root, split="val", seed=SEED)

        if len(val_dataset) == 0:
            record_result(6, desc, False, "Validation dataset is empty.")
            return

        # Subset to first 50 samples only for speed
        indices = list(range(min(50, len(val_dataset))))
        subset = torch.utils.data.Subset(val_dataset, indices)
        loader = torch.utils.data.DataLoader(subset, batch_size=4, shuffle=False, num_workers=0)

        class_names = ["no-damage", "minor-damage", "major-damage", "destroyed"]
        results = evaluate_baseline_a(model, loader, device, class_names)

        required_keys = ["accuracy", "macro_f1", "per_class_f1", "variance_score",
                         "confusion_matrix", "all_predictions", "all_labels", "all_softmax_probs"]

        for k in required_keys:
            if k not in results:
                record_result(6, desc, False, f"Missing key in results: {k}")
                return

        probs_shape = list(results["all_softmax_probs"].shape)
        if probs_shape != [len(indices), 4]:
            record_result(6, desc, False, f"all_softmax_probs shape incorrect: {probs_shape}")
            return

        record_result(6, desc, True, "Evaluation function returns correct structure")
    except Exception as e:
        record_result(6, desc, False, str(e))

test_6()


In [ ]:
def test_7():
    desc = "Visualisations"
    try:
        from src.evaluation.visualise import generate_architecture_diagram, generate_experimental_design_table

        arch_path = "../results/architecture_diagram.png"
        table_path = "../results/experimental_design_table.png"

        generate_architecture_diagram(arch_path)
        generate_experimental_design_table(table_path)

        if not os.path.exists(arch_path) or not os.path.exists(table_path):
            record_result(7, desc, False, "PNG files were not created.")
            return

        display(Image(filename=arch_path, width=400))
        display(Image(filename=table_path, width=600))

        record_result(7, desc, True, "Visualisations generated")
    except Exception as e:
        record_result(7, desc, False, str(e))

test_7()


In [ ]:
def test_8():
    desc = "DisasterM3 graceful failure"
    try:
        from src.preprocessing.filter_disasters import run_filter_pipeline
        from src.preprocessing.event_splitter import run_split_pipeline

        print("--- Testing filter pipeline (expected to print error but not crash) ---")
        run_filter_pipeline(disasterm3_root="/dummy/path/not/exist", output_dir="/dummy/out")

        print("\n--- Testing split pipeline (expected to print error but not crash) ---")
        run_split_pipeline(processed_dir="/dummy/path/not/exist", output_dir="/dummy/out")

        record_result(8, desc, True, "DisasterM3 scripts handle missing data correctly")
    except Exception as e:
        record_result(8, desc, False, f"Scripts crashed instead of failing gracefully: {e}")

test_8()


In [ ]:
def print_summary():
    print("\n" + "="*80)
    print("INTEGRATION TEST SUMMARY")
    print(f"| {'Test':<4} | {'Description':<30} | {'Result':<10} |")
    print("|------|--------------------------------|------------|")

    all_passed = True
    failed_tests = []

    for res in TEST_RESULTS:
        print(f"| {res['Test']:<4} | {res['Description']:<30} | {res['Result']:<10} |")
        if res['Result'] != "PASS":
            all_passed = False
            failed_tests.append(res)

    print("="*80 + "\n")

    if all_passed:
        print("✅ ALL TESTS PASSED — Ready for faculty meeting")
    else:
        print("❌ FAILURES DETECTED — Fix before faculty meeting\n")
        for res in failed_tests:
            print(f"Test {res['Test']} ({res['Description']}) failed:")
            print(f"  -> {res['Message']}\n")

print_summary()


## Faculty Meeting Checklist
1. Show `results/architecture_diagram.png`
2. Show `results/experimental_design_table.png`
3. Show `results/baselines/disaster_type_distribution.png` (from EDA)
4. Show `results/baselines/damage_class_distribution.png` (from EDA)
5. Show `results/baselines/sample_*.png` (pre/post image pair visualisations)
6. Show the integration test summary table from Cell 11
7. Show training smoke test loss output (proves pipeline runs)
8. State clearly: **"Full training will run overnight; these tests confirm the complete pipeline is working end-to-end"**